### **Notebook #3**

# Group 2: Luca Milani, Marta Laskowska, Monika Kaczorowska
## EDA - Stock Price Data

### Libraries

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math
import requests
import bs4 as bs
import yfinance as yf
import datetime
from scipy.stats import norm
from collections import Counter


from ipywidgets import interact, IntSlider, Checkbox
from functools import lru_cache
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)

### Dataset

In [17]:
def load_excel_sheets_except_request_table(
    file_path: str,
    date_col: str = "Name",
    skip_sheets=("REQUEST_TABLE",)
) -> pd.DataFrame:
    # Read all sheets into a dict: {sheet_name: DataFrame}
    book = pd.read_excel(file_path, sheet_name=None, engine="openpyxl")

    frames = []
    for sheet_name, df in book.items():
        if sheet_name in skip_sheets:
            continue
        if df.empty:
            continue

        # Ensure the date column exists; skip if missing
        if date_col not in df.columns:
            continue

        # Clean & standardize date
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
        frames.append(df)

    if not frames:
        return pd.DataFrame()  # nothing usable

    out = pd.concat(frames, ignore_index=True, sort=False)
    out = out.sort_values(date_col).reset_index(drop=True)
    return out

In [18]:
combined_df = load_excel_sheets_except_request_table("banks_data_bocconi/banks_ri.xlsm")
display(combined_df.head())

,Name,AAREAL BANK - TOT RETURN IND,COMDIRECT BANK - TOT RETURN IND,COMMERZBANK - TOT RETURN IND,DT.PFANDBRIEFBANK - TOT RETURN IND,PROCREDIT HOLDING - TOT RETURN IND,UMWELTBANK - TOT RETURN IND,ALLIANZ - TOT RETURN IND,DEUTSCHE BANK - TOT RETURN IND,BANQUE NATIONALE DE BELGIQUE - TOT RETURN IND,KBC ANCORA - TOT RETURN IND,KBC GROUP - TOT RETURN IND,BANK OF CYPRUS HOLDING - TOT RETURN IND,HELLENIC BANK - TOT RETURN IND,LHV GROUP - TOT RETURN IND,BBV.ARGENTARIA - TOT RETURN IND,BANCO DE SABADELL - TOT RETURN IND,BANCO SANTANDER - TOT RETURN IND,BANKIA - TOT RETURN IND,BANKINTER 'R' - TOT RETURN IND,CAIXABANK - TOT RETURN IND,LIBERBANK - TOT RETURN IND,UNICAJA BANCO - TOT RETURN IND,AKTIA BANK A - TOT RETURN IND,NORDEA BANK - TOT RETURN IND,BNP PARIBAS - TOT RETURN IND,CRCAM ILLE-VIL.CCI - TOT RETURN IND,CRCAM NORD CCI - TOT RETURN IND,CREDIT AGRICOLE - TOT RETURN IND,CREDIT AGRICOLE BRIE PICARDIE - TOT RETURN IND,CREDIT AGR.ILE DE FRANCE - TOT RETURN IND,CREDIT FONCIER DE MONACO - TOT RETURN IND,NATIXIS - TOT RETURN IND,SOCIETE GENERALE - TOT RETURN IND,BANQUE DE SAVOIE DEAD - 01/04/10 - TOT RETURN IND,CREDIT AGRICOLE - TOT RETURN IND.1,MERSEN (EX LCL) - TOT RETURN IND,ALPHA BANK - TOT RETURN IND,ATTICA BANK - TOT RETURN IND,BANK OF GREECE - TOT RETURN IND,BANK OF PIRAEUS - TOT RETURN IND,EUROBANK HOLDINGS - TOT RETURN IND,NATIONAL BK.OF GREECE - TOT RETURN IND,AIB GROUP - TOT RETURN IND,BANK OF IRELAND GROUP - TOT RETURN IND,PERMANENT TSB GHG. - TOT RETURN IND,BANCA GENERALI - TOT RETURN IND,BANCA MONTE DEI PASCHI - TOT RETURN IND,BCA.PICCOLO CDT.VALTELL - TOT RETURN IND,BANCA PPO.DI SONDRIO - TOT RETURN IND,BANCO BPM - TOT RETURN IND,BNC.DI DESIO E DELB. - TOT RETURN IND,BPER BANCA - TOT RETURN IND,CREDITO EMILIANO - TOT RETURN IND,FINECOBANK SPA - TOT RETURN IND,ILLIMITY BANK - TOT RETURN IND,INTESA SANPAOLO - TOT RETURN IND,UNICREDIT - TOT RETURN IND,UNIONE DI BANCHE ITALIAN - TOT RETURN IND,MEDIOBANCA BC.FIN - TOT RETURN IND,SIAULIU BANKAS - TOT RETURN IND,BANK OF VALLETTA - TOT RETURN IND,FIMBANK - TOT RETURN IND,HSBC BANK MALTA - TOT RETURN IND,LOMBARD BANK - TOT RETURN IND,ABN AMRO BANK - TOT RETURN IND,ING GROEP - TOT RETURN IND,NIBC HOLDING - TOT RETURN IND,VAN LANSCHOT KEMPEN - TOT RETURN IND,ASR NEDERLAND - TOT RETURN IND,AEGON - TOT RETURN IND,DELTA LLOYD GROUP DEAD - DEAD 01/06/17 - TOT RETURN IND,BANK FUR TIROL UND VBG. - TOT RETURN IND,BAWAG GROUP - TOT RETURN IND,BKS BANK - TOT RETURN IND,ERSTE GROUP BANK - TOT RETURN IND,OBERBANK - TOT RETURN IND,OBERBANK PREF. - TOT RETURN IND,RAIFFEISEN BANK INTL. - TOT RETURN IND,BANCO COMR.PORTUGUES 'R' - TOT RETURN IND,NOVA LJUBLJANSKA BANK DD NPV - TOT RETURN IND,OTP BANKA SLOVENSKO - TOT RETURN IND,OTP BANKA SLOVENSKO2 - TOT RETURN IND,OTP BANKA SLOVENSKO3 - TOT RETURN IND,TATRA BANKA - TOT RETURN IND,TATRA BANKA 2 - TOT RETURN IND,VSEOBECNA UVEROVA BANKA - TOT RETURN IND,BANK OF AMERICA - TOT RETURN IND,TRUIST FINANCIAL - TOT RETURN IND,BOK FINL. - TOT RETURN IND,CIT GROUP - TOT RETURN IND,CITIZENS FINANCIAL GROUP - TOT RETURN IND,COMERICA - TOT RETURN IND,COMMERCE BCSH. - TOT RETURN IND,CREDICORP - TOT RETURN IND,CULLEN FO.BANKERS - TOT RETURN IND,DISCOVER FINANCIAL SVS. - TOT RETURN IND,EAST WEST BANCORP - TOT RETURN IND,FIFTH THIRD BANCORP - TOT RETURN IND,FIRST CTZN.BCSH.A - TOT RETURN IND,FIRST FINL.BKSH. - TOT RETURN IND,FIRST HORIZON NATIONAL - TOT RETURN IND,FIRST REPUBLIC BANK - TOT RETURN IND,FNB - TOT RETURN IND,HUNTINGTON BCSH. - TOT RETURN IND,IBERIABANK - TOT RETURN IND,KEYCORP - TOT RETURN IND,M&T BANK - TOT RETURN IND,NEW YORK COMMUNITY BANCORP - TOT RETURN IND,PACWEST BANCORP - TOT RETURN IND,PEOPLES UNITED FINANCIAL - TOT RETURN IND,PINNACLE FINANCIAL PTNS. - TOT RETURN IND,PNC FINL.SVS.GP. - TOT RETURN IND,POPULAR - TOT RETURN IND,PROSPERITY BCSH. - TOT RETURN IND,REGIONS FINL.NEW - TOT RETURN IND,SIGNATURE BANK - TOT RETURN IND,STERLING BANCORP - TOT RETURN IND,SVB FINANCIAL GROUP - TOT RETURN IND,SYNOVUS FINANCIAL - TOT RETURN IND,TCF FINANCIAL - TO

In [7]:
bank_ri = pd.read_excel('banks_data_bocconi/banks_ri.xlsm')
bank_ri_scand = pd.read_excel('banks_data_bocconi/scandinavian_banks_ri.xlsm')
bank_pi = pd.read_excel('banks_data_bocconi/banks_pi.xlsm')
bank_pi_scand = pd.read_excel('banks_data_bocconi/scandinavian_banks_pi.xlsm')
fff_us = pd.read_excel('banks_data_bocconi/Europe_3_Factors_Daily.xlsx')
fff_eu = pd.read_excel('banks_data_bocconi/North_America_3_Factors_Daily.xlsx')
banks_info = pd.read_excel('banks_data_bocconi/banks_info.xlsx')
rfr_us = pd.read_excel('banks_data_bocconi/risk_free_rate.xlsm')

In [16]:
bank_ri.head(5)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,3.0.14.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Last Refreshed:,29/04/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Update,Request Type,Format,Series Lookup,Datatype/Expressions,Start Date,End Date,Freq,Reserved for Chart Name,NaN,Status,Description,NaN,NaN,Statistics,NaN,NaN,NaN,Sequencing,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rows,Cols,Total,Started,Finished,Duration,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
bank_ri_scand.head(2)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,3.0.14.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
bank_pi.head(2)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,3.0.14.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
bank_pi_scand.head(2)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,3.0.14.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
fff_us.head(2)

,date,Mkt-RF,SMB,HML,RF
0,07/02/1990,0.99,0.05,-0.53,0.03
1,07/03/1990,0.33,-0.12,-0.03,0.03


In [13]:
fff_eu.head(2)

,date,Mkt-RF,SMB,HML,RF
0,07/02/1990,0.30,-0.38,-0.12,0.03
1,07/03/1990,0.14,-0.08,-0.40,0.03


In [14]:
banks_info.head(2)

,Banks Emerging Markets,Unnamed: 1,Unnamed: 2
0,LBANKSEM,NaN,LISTE RETENUE
1,Market,Europe (EMU only),NaN


In [15]:
rfr_us.head(2)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,3.0.14.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
